# Experiment 1 configuration

## Objective and search domain

For $x \in [-4,4]^2$ and $c \ge 0$, minimise

$$
f_c(x) = \min\left\{
    1 + \lVert x-a \rVert_2^2,\;
    \lVert x-b \rVert_2^2
    + c\left[\max\left(0, \lVert x-b \rVert_2 - 0.5\right)\right]^2
\right\},
\qquad a = (-2,0),\quad b = (2,0).
$$

Basin A has a local minimum of 1 at $a$. Basin B has the global minimum
of 0 at $b$. Increasing $c$ shrinks the region where $f_c(x)<1$ and
steepens the B branch outside its core. The radius-0.5 core and the
objective inside it stay the same. The iteration changes the objective
values outside the core as well as the size of the discovery region.

In [ ]:
import numpy as np

# A is the inferior minimum (value 1); B is the global minimum (value 0).
BASIN_A = np.array([-2.0, 0.0])
BASIN_B = np.array([2.0, 0.0])
CORE_RADIUS = 0.5
DOMAIN_BOUNDS = np.array([[-4.0, 4.0], [-4.0, 4.0]])
PENALTY_VALUES = [0.0, 3.0, 10.0, 30.0]


def two_basin_function(x, c, a=BASIN_A, b=BASIN_B, r0=CORE_RADIUS):
    """Return the lower of the two basin values at a two-coordinate point x."""
    x = np.asarray(x, dtype=float)
    distance_a = np.linalg.norm(x - a)
    distance_b = np.linalg.norm(x - b)
    basin_a = 1.0 + distance_a**2
    # The extra penalty applies only outside B's radius-r0 core.
    basin_b = distance_b**2 + c * max(0.0, distance_b - r0)**2
    # Taking the lower branch joins the two basins into one objective.
    return float(min(basin_a, basin_b))


def make_objective(c, a=BASIN_A, b=BASIN_B, r0=CORE_RADIUS):
    """Fix the basin settings and return a function that accepts only a point."""

    # check inputs
    if not np.isfinite(c) or c < 0 or not np.isfinite(r0) or r0 <= 0:
        raise ValueError("c must be nonnegative and r0 must be positive and finite")

    # Capture these settings so every optimiser receives a function of x alone.
    def objective(x):
        return two_basin_function(x, c, a, b, r0)

    return objective

## Run settings

BO uses expected improvement with four uniform initial evaluations.
CMA-ES uses a uniform initial mean, an initial step size of 0.25 in
unit coordinates, and the PyCMA default population size. Set
`COMMON_CMA["popsize"]` to an integer to change it. TuRBO-1 uses four
initial evaluations and a batch size of one. Random samples uniformly
throughout.

The default iteration uses $c \in \{0,3,10,30\}$, seeds 0–9, and 50 evaluations
per method per seed. The four strict targets are $f<1$, $f<0.25$,
$f<10^{-2}$, and $f<10^{-4}$. The concentration diagnostic measures the
fraction of the last 20 evaluations within distance 1 of $a$.

In [ ]:
# Shared method settings are reused at every penalty and seed for a fair comparison.
COMMON_BO = {
    "n_initial_points": 4,
    "initial_point_generator": "random",
    "acq_func": "EI",
    # The objective is deterministic; this tiny noise value helps numerical stability.
    "noise": 1e-10,
    "xi": 0.0,
}
# sigma0 is measured in unit coordinates; None uses CMA-ES's default population size.
COMMON_CMA = {"sigma0": 0.25, "seed_offset": 1, "popsize": None}
# TuRBO starts with four points and then proposes one point at a time.
COMMON_TURBO = {
    "n_initial_points": 4,
    "batch_size": 1,
    "use_ard": True,
    "n_training_steps": 50,
    "verbose": False,
    "device": "cpu",
    "dtype": "float64",
}
METHOD_PARAMS = {"BO": COMMON_BO, "CMA-ES": COMMON_CMA, "TuRBO-1": COMMON_TURBO, "Random": {}}
# Strict targets measure discovery, core entry, and increasingly precise refinement.
ANALYSIS_SETTINGS = {
    "targets": [1.0, 0.25, 1e-2, 1e-4],
    # Measure concentration near A using the final 20 evaluations within distance 1.
    "near_a_radius": 1.0,
    "late_evaluations": 20,
}
# Run every penalty/seed/method combination: 4 x 10 x 4 = 160 trials by default.
RUN_SETTINGS = {
    "c_values": PENALTY_VALUES,
    "seeds": list(range(10)),
    "methods": list(METHOD_PARAMS),
    # Each trial gets 50 objective calls, including initial points and any restarts.
    "budget": 50,
    # Use plain lists so the complete experiment configuration can be saved as JSON.
    "problem": {
        "a": BASIN_A.tolist(),
        "b": BASIN_B.tolist(),
        "r0": CORE_RADIUS,
        "bounds": DOMAIN_BOUNDS.tolist(),
    },
    "method_params": METHOD_PARAMS,
}